# Procurement Spend Analytics — EDA
**Notebook:** `01_eda.ipynb`  
**Purpose:** Exploratory Data Analysis of the raw procurement spend dataset.

**Sections**
1. Load Procurement Dataset  
2. Inspect Data Structure  
3. Profile Suppliers  
4. Check Missing Values  
5. Visualize Missing Data  


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # make app/ importable

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


## 1. Load Procurement Dataset
Load `data/raw/procurement_spend_raw.csv` through the ingestion pipeline in `app/ingest/load_data.py`, which casts types and attaches data-quality flags.


In [ ]:
from app.ingest.load_data import load_and_prepare

RAW_PATH = os.path.abspath("../data/raw/procurement_spend_raw.csv")
df = load_and_prepare(RAW_PATH)

print(f"Dataset shape: {df.shape}")
df.head()


## 2. Inspect Data Structure
Examine column types, shapes, and descriptive statistics.


In [ ]:
print("=== df.info() ===")
df.info()


In [ ]:
print("=== Column dtypes ===")
print(df.dtypes)


In [ ]:
print("=== Descriptive Statistics (numeric) ===")
df.describe(include=[np.number])


## 3. Profile Suppliers
Count distinct suppliers, total and average spend per supplier, and identify the top 10 by spend volume.


In [ ]:
# Total unique suppliers
n_suppliers = df["supplier_id"].nunique()
print(f"Total distinct suppliers: {n_suppliers}")

# Supplier spend profile
supplier_profile = (
    df.groupby(["supplier_id", "supplier_name"], dropna=False)
    .agg(
        po_count=("po_id", "count"),
        total_spend=("amount_usd", "sum"),
        avg_spend=("amount_usd", "mean"),
        categories=("category", lambda x: ", ".join(sorted(x.dropna().unique()))),
    )
    .reset_index()
    .sort_values("total_spend", ascending=False)
)

print(f"\nSupplier profile ({len(supplier_profile)} rows):")
supplier_profile.head(10)


In [ ]:
# Top 10 suppliers by total spend — bar chart
top10 = supplier_profile.head(10).copy()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top10["supplier_name"], top10["total_spend"], color=sns.color_palette("Blues_r", 10))
ax.set_xlabel("Total Spend (USD)")
ax.set_title("Top 10 Suppliers by Total Spend")
ax.invert_yaxis()
for bar in bars:
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
            f"${bar.get_width():,.0f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Supplier count by category
supplier_by_category = (
    df.groupby("category")["supplier_id"].nunique()
    .reset_index()
    .rename(columns={"supplier_id": "unique_suppliers"})
    .sort_values("unique_suppliers", ascending=False)
)
print("Unique suppliers per category:")
print(supplier_by_category.to_string(index=False))


## 4. Check Missing Values
Identify columns with missing data and quantify the extent.


In [ ]:
# Only consider the core columns (exclude engineered flag columns)
core_cols = [c for c in df.columns if not c.startswith("flag_")]
df_core = df[core_cols]

missing_count = df_core.isnull().sum()
missing_pct = (missing_count / len(df_core) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct,
}).sort_values("missing_count", ascending=False)

print("Missing value summary:")
print(missing_summary[missing_summary["missing_count"] > 0].to_string())


In [ ]:
# Data quality flag summary from ingestion pipeline
flag_cols = [c for c in df.columns if c.startswith("flag_")]
flag_summary = df[flag_cols].sum().rename("count")
flag_summary["pct"] = (flag_summary["count"] / len(df) * 100).round(2)
print("Data quality flags:")
print(flag_summary.to_string())


## 5. Visualize Missing Data
Heatmap of nulls across rows and columns to reveal structural patterns.


In [ ]:
# Seaborn heatmap of null values (True = missing)
null_matrix = df_core.isnull()

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    null_matrix,
    cbar=True,
    cmap="YlOrRd",
    yticklabels=False,
    ax=ax,
    cbar_kws={"label": "Missing (1 = null)"},
)
ax.set_title("Missing Value Heatmap — Procurement Spend Dataset", fontsize=13)
ax.set_xlabel("Columns")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Bar chart of missing % per column
cols_with_missing = missing_summary[missing_summary["missing_count"] > 0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(cols_with_missing.index, cols_with_missing["missing_pct"], color=sns.color_palette("Reds_r", len(cols_with_missing)))
ax.set_ylabel("Missing %")
ax.set_title("Missing Value % by Column")
ax.axhline(y=5, color="grey", linestyle="--", linewidth=0.8, label="5% threshold")
ax.legend()
plt.xticks(rotation=30, ha="right", fontsize=9)
for i, (col, row) in enumerate(cols_with_missing.iterrows()):
    ax.text(i, row["missing_pct"] + 0.3, f"{row['missing_pct']}%", ha="center", fontsize=8)
plt.tight_layout()
plt.show()
